In [ ]:
!pip install -q transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.6 MB/s eta 0:00:00


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# ---------------- MODEL LOAD ----------------
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

torch.backends.cuda.matmul.allow_tf32 = True

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16,
    use_cache=True
)

model.eval()

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

In [ ]:
# ---------------- LLM FUNCTION ----------------
def generate_response(messages):

    prompt = ""

    for msg in messages:
        if msg["role"] == "system":
            prompt += f"{msg['content']}\n"
        elif msg["role"] == "user":
            prompt += f"<s>[INST] {msg['content']} [/INST]"
        else:
            prompt += f"{msg['content']}</s>"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.1
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return response.split("[/INST]")[-1].strip()

In [ ]:

# ---------------- CHAT LOOP ----------------
print("🤖 Smart Deal Chatbot Ready! Type 'quit' to exit.\n")

messages = [
    {
        "role": "system",
        "content": "You are a smart shopping assistant. You help users find cheapest prices, best deals, and coupons. Give structured and useful answers."
    }
]

MAX_HISTORY = 4

while True:
    user_input = input("You: ")

    if user_input.lower() == "quit":
        print("Bot: Take care 😊")
        break

    messages.append({"role": "user", "content": user_input})

    # KEEP system + last messages
    messages = [messages[0]] + messages[-MAX_HISTORY:]

    # smart_reply not added yet
    response = generate_response(messages)

    messages.append({"role": "assistant", "content": response})

    print("Bot:", response)

🤖 Smart Deal Chatbot Ready! Type 'quit' to exit.

You: jg


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 119.5 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

Writing app.py


In [ ]:
# ---------------- INTENT ----------------
def detect_intent(query):
    query = query.lower()

    if any(x in query for x in ["price", "cost", "cheapest"]):
        return "price"
    elif any(x in query for x in ["deal", "best", "under"]):
        return "deals"
    elif any(x in query for x in ["coupon", "offer", "promo"]):
        return "coupon"
    else:
        return "chat"

In [ ]:
# ---------------- CLEAN QUERY ----------------
def clean_query(query):
    words_to_remove = ["price", "cost", "cheapest", "buy"]

    for w in words_to_remove:
        query = query.replace(w, "")

    return query.strip()

In [ ]:
!pip install beautifulsoup4

In [ ]:
import requests
from bs4 import BeautifulSoup

def get_amazon_price(query):
    query = clean_query(query)

    url = f"https://www.amazon.in/s?k={query.replace(' ', '+')}"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        r = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(r.text, "html.parser")

        items = soup.select(".s-result-item")

        for item in items:
            title = item.select_one("h2")
            price = item.select_one(".a-price-whole")

            if title and price:
                return {
                    "site": "Amazon",
                    "title": title.text.strip(),
                    "price": price.text.strip()
                }

    except:
        return None

    return None

In [ ]:
# ---------------- COUPON SCRAPER ----------------
def get_coupons(query):

    # extract site intelligently
    query = query.lower()

    known_sites = ["myntra", "amazon", "flipkart", "ajio"]

    site = None
    for s in known_sites:
        if s in query:
            site = s
            break

    if not site:
        return []

    url = f"https://www.grabon.in/{site}-coupons/"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        r = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(r.text, "html.parser")

        coupons = []

        for item in soup.select(".coupon-code")[:5]:
            coupons.append(item.text.strip())

        return coupons

    except:
        return []

In [ ]:
# ---------------- CONTROLLER ----------------
def smart_reply(user_input):
    intent = detect_intent(user_input)

    # PRICE
    if intent == "price":
        data = get_amazon_price(user_input)

        if data:
            return f"""
🛒 {data['title']}
💰 ₹{data['price']} on {data['site']}
"""
        else:
            return "Couldn't find price right now."

    # COUPONS
    elif intent == "coupon":
        coupons = get_coupons(user_input)

        if coupons:
            return "Coupons:\n" + "\n".join(coupons)
        else:
            return "❌ No coupons found."

    return None

In [ ]:
%%writefile app.py

import streamlit as st
import torch
import requests
from bs4 import BeautifulSoup
from transformers import AutoModelForCausalLM, AutoTokenizer


# ---------------- INTENT ----------------
def detect_intent(query):
    query = query.lower()

    if any(x in query for x in ["price", "cost", "cheapest"]):
        return "price"
    elif any(x in query for x in ["deal", "best", "under"]):
        return "deals"
    elif any(x in query for x in ["coupon", "offer", "promo"]):
        return "coupon"
    else:
        return "chat"


# ---------------- CLEAN QUERY ----------------
def clean_query(query):
    for w in ["price", "cost", "cheapest", "buy"]:
        query = query.replace(w, "")
    return query.strip()


# ---------------- AMAZON ----------------
def get_amazon_price(query):
    query = clean_query(query)
    url = f"https://www.amazon.in/s?k={query.replace(' ', '+')}"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        r = requests.get(url, headers=headers, timeout=5)
        soup = BeautifulSoup(r.text, "html.parser")

        for item in soup.select(".s-result-item"):
            title = item.select_one("h2")
            price = item.select_one(".a-price-whole")

            if title and price:
                return f"{title.text.strip()} - ₹{price.text.strip()} (Amazon)"
    except:
        return None

    return None


# ---------------- COUPONS ----------------
def get_coupons(query):
    sites = ["myntra", "amazon", "flipkart"]

    site = None
    for s in sites:
        if s in query.lower():
            site = s

    if not site:
        return []

    url = f"https://www.grabon.in/{site}-coupons/"
    headers = {"User-Agent": "Mozilla/5.0"}

    try:
        r = requests.get(url, headers=headers)
        soup = BeautifulSoup(r.text, "html.parser")

        return [c.text.strip() for c in soup.select(".coupon-code")[:5]]
    except:
        return []


# ---------------- CONTROLLER ----------------
def smart_reply(user_input):
    intent = detect_intent(user_input)

    if intent == "price":
        result = get_amazon_price(user_input)
        return result if result else None

    elif intent == "coupon":
        coupons = get_coupons(user_input)
        return "\n".join(coupons) if coupons else None

    return None


# ---------------- LOAD MODEL ----------------
@st.cache_resource
def load_model():
    model_name = "mistralai/Mistral-7B-Instruct-v0.1"

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16,
        use_cache=True
    )

    model.eval()
    return tokenizer, model


tokenizer, model = load_model()


# ---------------- LLM ----------------
def generate_response(messages):
    prompt = ""

    for msg in messages:
        if msg["role"] == "system":
            prompt += msg["content"] + "\n"
        elif msg["role"] == "user":
            prompt += f"<s>[INST] {msg['content']} [/INST]"
        else:
            prompt += f"{msg['content']}</s>"

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split("[/INST]")[-1].strip()


# ---------------- UI ----------------
st.title("🛒 Smart Deal Finder Bot")

if "messages" not in st.session_state:
    st.session_state.messages = [
        {"role": "system", "content": "You are a smart shopping assistant."}
    ]

for msg in st.session_state.messages:
    if msg["role"] != "system":
        with st.chat_message(msg["role"]):
            st.write(msg["content"])

user_input = st.chat_input("Ask about deals, prices, coupons...")

if user_input:
    st.session_state.messages.append({"role": "user", "content": user_input})

    with st.chat_message("user"):
        st.write(user_input)

    custom = smart_reply(user_input)

    if custom:
        response = custom
    else:
        response = generate_response(st.session_state.messages[-4:])

    st.session_state.messages.append({"role": "assistant", "content": response})

    with st.chat_message("assistant"):
        st.write(response)

Overwriting app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹

⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.87.106.161:8501

y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼your url is: https://sharp-cobras-knock.loca.lt
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [01:03<00:00,  4.60it/s, Materializing param=model.norm.weight]
Some parameters are on the meta device because they were offloaded to the cpu and disk.
  Stopping...
^C


In [ ]:
!killall streamlit

streamlit: no process found


In [ ]:
!streamlit run app.py &




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.87.106.161:8501

  Stopping...


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501



⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸your url is: https://all-bottles-eat.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.87.106.161:8501

y
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100% 291/291 [01:06<00:00,  4.40it/s, Materializing param=model.norm.weight]
Some parameters are on the meta device because they were offloaded to the disk and cpu.
